In [ ]:
import os
import re
from pathlib import Path

import pandas as pd


# ============================================================================
# 01. DATA PREPROCESSING
# ============================================================================
# Run from notebooks/ in the nli_project folder.

BASE_DIR = Path("..").resolve()

TOEFL_TRAIN_PATH = BASE_DIR / "data/raw/TOEFL11/toefl_train.csv"
TOEFL_TEST_PATH = BASE_DIR / "data/raw/TOEFL11/toefl_test.csv"
ICLE_METADATA_PATH = BASE_DIR / "data/raw/ICLEV3/metadata.csv"
ICLE_TEXTS_DIR = BASE_DIR / "data/raw/ICLEV3/texts"

OUTPUT_RAW_PATH = BASE_DIR / "data/raw/combined_nli_raw.csv"
OUTPUT_AUDIT_PATH = BASE_DIR / "data/raw/combined_nli_raw_audit.csv"

TARGET_LANGS = ["DEU", "FRA", "ITA", "JPN", "KOR", "SPA", "TUR", "ZHO"]
ICLE_LANG_MAP = {
    "German": "DEU",
    "French": "FRA",
    "Italian": "ITA",
    "Japanese": "JPN",
    "Korean": "KOR",
    "Spanish": "SPA",
    "Turkish": "TUR",
    "Chinese": "ZHO",
}

# ICLEv3's own raw .txt files carry a leading corpus document-ID header, e.g.
# "<ICLE-FR-UCL-0027.2>" -- country code + institution code + file number.
# This is corpus bookkeeping, not essay content, but clean_text() previously
# passed it straight through into the "text" column untouched, where it
# directly encoded each essay's own country/institution -- confirmed present
# in 100% of raw ICLEv3 texts (data/processed/results/... header-leak check),
# and confirmed absent from TOEFL11 text, so stripping it is a no-op there.
ICLE_HEADER_PATTERN = re.compile(r"^\s*<ICLE-[A-Za-z]{2}-[A-Za-z0-9\-]+-\d+\.\d+>\s*")


def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")


def clean_text(text: str) -> str:
    """Normalize whitespace while preserving paragraph/sentence boundary hints.
    Also strips a leading ICLE corpus document-ID header (e.g.
    "<ICLE-FR-UCL-0027.2>"), if present -- see ICLE_HEADER_PATTERN above for
    why. No-op for TOEFL11 text, which never contains this pattern."""
    if not isinstance(text, str):
        return ""

    text = ICLE_HEADER_PATTERN.sub("", text, count=1)
    text = text.replace("\xa0", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n\s*\n+", " <P> ", text)
    text = re.sub(r"\n+", " <S> ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def validate_columns(df: pd.DataFrame, required_columns: set[str], source_name: str) -> None:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{source_name} is missing required columns: {sorted(missing)}")


print("Loading TOEFL11 data...")
require_file(TOEFL_TRAIN_PATH)
require_file(TOEFL_TEST_PATH)

toefl_train = pd.read_csv(TOEFL_TRAIN_PATH)
toefl_test = pd.read_csv(TOEFL_TEST_PATH)
validate_columns(toefl_train, {"text", "Language"}, "TOEFL train")
validate_columns(toefl_test, {"text", "Language"}, "TOEFL test")

toefl_train = toefl_train.assign(split="train")
toefl_test = toefl_test.assign(split="test")
toefl_full = pd.concat([toefl_train, toefl_test], ignore_index=True)
toefl_filtered = toefl_full[toefl_full["Language"].isin(TARGET_LANGS)].copy()

toefl_master = pd.DataFrame(
    {
        "doc_id": [f"TOEFL11_{i:05d}" for i in range(len(toefl_filtered))],
        "text": toefl_filtered["text"].map(clean_text),
        "label": toefl_filtered["Language"].astype(str).str.strip().str.upper(),
        "dataset": "TOEFL11",
        "split": toefl_filtered["split"].astype(str),
        "source_file": "",
    }
)

print(f"TOEFL11 kept: {len(toefl_master)} essays")


print("Loading ICLEv3 data...")
require_file(ICLE_METADATA_PATH)
if not ICLE_TEXTS_DIR.exists():
    raise FileNotFoundError(f"ICLE texts directory not found: {ICLE_TEXTS_DIR}")

icle_meta = pd.read_csv(ICLE_METADATA_PATH)
validate_columns(icle_meta, {"Native language", "File name"}, "ICLE metadata")
icle_meta = icle_meta[icle_meta["Native language"].isin(ICLE_LANG_MAP)].copy()

icle_rows = []
missing_files = []

for _, row in icle_meta.iterrows():
    file_id = str(row["File name"]).strip()
    file_name = file_id if file_id.endswith(".txt") else f"{file_id}.txt"
    file_path = ICLE_TEXTS_DIR / file_name

    if not file_path.exists():
        missing_files.append(file_name)
        continue

    raw_text = file_path.read_text(encoding="utf-8", errors="ignore")
    icle_rows.append(
        {
            "doc_id": f"ICLEV3_{Path(file_name).stem}",
            "text": clean_text(raw_text),
            "label": ICLE_LANG_MAP[row["Native language"]],
            "dataset": "ICLEV3",
            "split": "external_test",
            "source_file": file_name,
        }
    )

icle_master = pd.DataFrame(icle_rows)
print(f"ICLEv3 kept: {len(icle_master)} essays; missing files: {len(missing_files)}")


master_df = pd.concat([toefl_master, icle_master], ignore_index=True)
master_df = master_df[master_df["text"].str.len() > 0].copy()

if master_df["doc_id"].duplicated().any():
    duplicates = master_df.loc[master_df["doc_id"].duplicated(), "doc_id"].head().tolist()
    raise ValueError(f"Duplicate doc_id values found, e.g. {duplicates}")

OUTPUT_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
master_df.to_csv(OUTPUT_RAW_PATH, index=False)

audit = (
    master_df.groupby(["dataset", "split", "label"])
    .size()
    .rename("n")
    .reset_index()
    .sort_values(["dataset", "split", "label"])
)
audit.to_csv(OUTPUT_AUDIT_PATH, index=False)

print(f"Saved combined corpus: {OUTPUT_RAW_PATH}")
print(f"Saved audit table: {OUTPUT_AUDIT_PATH}")
print(audit.pivot_table(index=["dataset", "split"], columns="label", values="n", fill_value=0))
